# Housekeeping: cloud-first LingBot-Map trial

This experiment runs reconstruction on a Google Colab GPU. It never connects to the household database. Source is pinned; no API key or paid endpoint is needed.

**Runtime → Change runtime type → T4 GPU**, if free capacity is offered. Stay on the free tier. If Colab offers only an upgrade, stop rather than buy compute. [Free-tier limits](https://research.google.com/colaboratory/faq.html).

Start with a slow, continuous 15–30 second walkthrough of shared areas, up to 3 minutes; the trial reconstructs the first 30 seconds. Avoid people, documents and private conversations. Uploading sends the file to Google Colab. Use a private notebook; do not save outputs containing personal imagery in public GitHub.

Geometry does not identify rooms, measure an accurate floor plan or verify cleaning. The app asks you to label areas and fixtures before producing an editable draft.


In [ ]:
import subprocess, sys, pathlib
subprocess.run(['nvidia-smi'], check=True)
repo=pathlib.Path('/content/lingbot-map')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/Robbyant/lingbot-map.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','8fdf984a7f9caf391622ea5843a8410900d27ef2'],check=True)
subprocess.run([sys.executable,'-m','pip','install','torch==2.8.0','torchvision==0.23.0','--index-url','https://download.pytorch.org/whl/cu128'],check=True)
subprocess.run([sys.executable,'-m','pip','install','numpy==2.2.6','opencv-python==4.11.0.86','-e',str(repo)],check=True)
from huggingface_hub import hf_hub_download
weights=hf_hub_download('robbyant/lingbot-map','lingbot-map.pt',revision='204754b')
print('Pinned source and checkpoint ready.')


## Choose the recording

Upload one short MP4 from your device. The recording remains in this temporary Colab session; no Drive mount or GitHub authorization is required.


In [ ]:
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1:
    raise ValueError('Upload one video.')
video=next(iter(uploaded))
if len(uploaded[video])>250*1024*1024:
    raise ValueError('Use a video smaller than 250 MB.')
del uploaded


## Benchmark and export

Choose PROFILE="fast" for the initial low-memory trial (336 pixels, 2 fps, two scale frames), or PROFILE="detailed" for 518 pixels, 4 fps, eight scale frames and four camera refinement steps. Both use SDPA and CPU output offload. The 120-frame cap limits a 4 fps trial to 30 seconds. These resource limits differ from paper benchmarks. If memory runs out, reconnect the runtime and retry with IMAGE_SIZE=252 and FPS=1. Do not interpret reconstruction gaps as unclean areas.


In [ ]:
import pathlib
_exporter_source="\"\"\"Bounded LingBot-Map benchmark/exporter. Runs on a GPU, outside Vercel.\n\nThe notebook pins upstream source and weights. This adapter does not execute\npickled model code, open a server, access the household DB, or label rooms.\n\"\"\"\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport time\n\nUPSTREAM_REVISION = \"8fdf984a7f9caf391622ea5843a8410900d27ef2\"\nMODEL_REVISION = \"204754b\"\n\n\ndef extract_frames(video, folder, fps=2, limit=120, clip_seconds=30):\n    import cv2\n    cap = cv2.VideoCapture(str(video))\n    source_fps = cap.get(cv2.CAP_PROP_FPS)\n    total = cap.get(cv2.CAP_PROP_FRAME_COUNT)\n    if not cap.isOpened() or not math.isfinite(source_fps) or source_fps <= 0 or total <= 0:\n        cap.release()\n        raise ValueError(\"Cannot decode video duration; use a supported MP4 recording.\")\n    source_duration = total / source_fps\n    if not 0 < source_duration <= 180 or not 0 < clip_seconds <= 60:\n        cap.release()\n        raise ValueError(\"Use a recording up to 3 minutes and a trial segment up to 60 seconds.\")\n    # The browser accepts at most 120 cameras; report only the sampled segment.\n    duration = min(source_duration, clip_seconds, limit / fps)\n    paths, times = [], []\n    try:\n        for i in range(min(limit, math.ceil(duration * fps))):\n            seconds = i / fps\n            cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)\n            ok, frame = cap.read()\n            if not ok:\n                break\n            path = folder / f\"{i:04d}.jpg\"\n            if not cv2.imwrite(str(path), frame):\n                raise ValueError(\"Could not prepare a video frame.\")\n            paths.append(str(path))\n            times.append(seconds)\n    finally:\n        cap.release()\n    if len(paths) < 2:\n        raise ValueError(\"Use a continuous clip with at least two readable frames.\")\n    return paths, times, duration\n\n\ndef compact_map(points, confidence, colors, camera_positions, times, duration, metrics, maximum=20000):\n    import numpy as np\n    points = np.asarray(points).reshape(-1, 3)\n    confidence = np.asarray(confidence).reshape(-1)\n    colors = np.asarray(colors).reshape(-1, 3)\n    cameras = np.asarray(camera_positions)\n    if points.shape != colors.shape or len(confidence) != len(points) or cameras.shape != (len(times), 3):\n        raise ValueError(\"Unexpected prediction shapes from the model.\")\n    valid = np.isfinite(points).all(axis=1) & np.isfinite(confidence)\n    if not valid.any() or not np.isfinite(cameras).all():\n        raise ValueError(\"Reconstruction contains no finite geometry.\")\n    valid &= confidence >= np.percentile(confidence[valid], 60)\n    points, colors = points[valid], colors[valid]\n    # Relative coordinates only: monocular scale is not a measured floor plan.\n    center = np.median(points, axis=0)\n    radius = np.linalg.norm(points - center, axis=1)\n    valid = radius <= np.percentile(radius, 98)\n    points, colors = points[valid], colors[valid]\n    scale = max(float(np.percentile(radius, 90)), 1e-6)\n    # Uniform sampling avoids regular stripes from flattened image rows.\n    indices = np.random.default_rng(42).choice(len(points), min(maximum, len(points)), replace=False)\n    points = np.round((points[indices] - center) / scale, 4)\n    colors = np.clip(colors[indices], 0, 255).astype(int)\n    cameras = np.round((cameras - center) / scale, 4)\n    if not len(points) or not np.isfinite(points).all() or np.max(np.abs(cameras)) > 100000:\n        raise ValueError(\"The reconstruction collapsed. Try a slower continuous recording.\")\n    return {\n        \"format\": \"housekeeping-map-v1\", \"model\": \"lingbot-map\",\n        \"revision\": UPSTREAM_REVISION, \"duration\": round(duration, 4),\n        \"points\": np.column_stack((points, colors)).tolist(),\n        \"cameras\": [{\"seconds\": round(t, 4), \"position\": p.tolist()} for t, p in zip(times, cameras)],\n        \"metrics\": metrics,\n    }\n\n\ndef run(args):\n    import numpy as np\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"A GPU runtime is required for this trial; no CPU fallback is started.\")\n    repo = Path(args.repo).resolve()\n    import subprocess\n    revision = subprocess.check_output([\"git\", \"-C\", str(repo), \"rev-parse\", \"HEAD\"], text=True).strip()\n    if revision != UPSTREAM_REVISION:\n        raise RuntimeError(\"Use the pinned LingBot-Map source revision from the trial notebook.\")\n    sys.path.insert(0, str(repo))\n    from lingbot_map.models.gct_stream import GCTStream\n    from lingbot_map.utils.load_fn import load_and_preprocess_images\n    from lingbot_map.utils.pose_enc import pose_encoding_to_extri_intri\n    from lingbot_map.utils.geometry import unproject_depth_map_to_point_map\n    with tempfile.TemporaryDirectory(prefix=\"mapping-frames-\") as folder:\n        paths, times, duration = extract_frames(args.video, Path(folder), args.fps,\n                                               clip_seconds=getattr(args, \"clip_seconds\", 30))\n        images = load_and_preprocess_images(paths, mode=\"crop\", image_size=args.image_size, patch_size=14)\n        detailed = getattr(args, \"profile\", \"fast\") == \"detailed\"\n        scale_frames = min(8 if detailed else 2, len(paths))\n        cache_window = 32 if detailed else 16\n        iterations = 4 if detailed else 1\n        keyframe_interval = 1 if detailed else 2\n        model = GCTStream(img_size=518, patch_size=14, enable_3d_rope=True,\n                          max_frame_num=1024, kv_cache_sliding_window=cache_window,\n                          kv_cache_scale_frames=scale_frames, kv_cache_cross_frame_special=True,\n                          kv_cache_include_scale_frames=True, use_sdpa=True, camera_num_iterations=iterations)\n        checkpoint = torch.load(args.weights, map_location=\"cpu\", weights_only=True)\n        state = checkpoint.get(\"model\", checkpoint)\n        model.load_state_dict(state, strict=True)\n        del checkpoint, state\n        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16\n        model.aggregator.to(dtype=dtype)\n        model = model.to(\"cuda\").eval()\n        images = images.to(\"cuda\")\n        torch.cuda.reset_peak_memory_stats()\n        torch.cuda.synchronize()\n        started = time.perf_counter()\n        with torch.inference_mode(), torch.amp.autocast(\"cuda\", dtype=dtype):\n            result = model.inference_streaming(images, num_scale_frames=scale_frames, keyframe_interval=keyframe_interval,\n                                               output_device=torch.device(\"cpu\"))\n        torch.cuda.synchronize()\n        elapsed = time.perf_counter() - started\n        peak = torch.cuda.max_memory_allocated() / (1024 * 1024)\n        pose = result[\"pose_enc\"].float().cpu()\n        extrinsic, intrinsic = pose_encoding_to_extri_intri(pose, images.shape[-2:])\n        extrinsic = extrinsic.cpu().numpy().reshape(-1, 3, 4)\n        intrinsic = intrinsic.cpu().numpy().reshape(-1, 3, 3)\n        world_to_camera = np.tile(np.eye(4), (len(extrinsic), 1, 1))\n        world_to_camera[:, :3, :4] = extrinsic\n        positions = np.linalg.inv(world_to_camera)[:, :3, 3]\n        colors = images.detach().float().cpu().permute(0, 2, 3, 1).numpy() * 255\n        # The pinned streaming checkpoint predicts depth, not a direct point map.\n        # Match the upstream viewer's depth-to-world reconstruction path.\n        depth = result[\"depth\"].float().cpu().numpy().reshape(len(times), *images.shape[-2:], 1)\n        points = unproject_depth_map_to_point_map(depth, extrinsic, intrinsic)\n        confidence = result[\"depth_conf\"].float().cpu().numpy()\n        metrics = {\"inferenceSeconds\": round(elapsed, 3), \"peakGpuMb\": round(peak, 1), \"frames\": len(times)}\n        output = compact_map(points, confidence, colors, positions, times, duration, metrics,\n                             maximum=30000 if detailed else 20000)\n        encoded = json.dumps(output, separators=(\",\", \":\"), allow_nan=False)\n        if len(encoded.encode(\"utf-8\")) > 4 * 1024 * 1024:\n            raise RuntimeError(\"Export exceeds the bounded browser import size.\")\n        Path(args.output).write_text(encoded, encoding=\"utf-8\")\n        report = {\"gpu\": torch.cuda.get_device_name(), \"sourceRevision\": revision,\n                  \"modelRevision\": MODEL_REVISION, \"reconstruction\": \"depth_unprojected\",\n                  \"profile\": getattr(args, \"profile\", \"fast\"), \"imageSize\": args.image_size,\n                  \"sampleFps\": args.fps, \"segmentSeconds\": duration,\n                  \"scaleFrames\": scale_frames, \"cameraIterations\": iterations,\n                  \"cacheWindow\": cache_window, \"keyframeInterval\": keyframe_interval,\n                  **metrics, \"points\": len(output[\"points\"]), \"bytes\": len(encoded)}\n        Path(args.output).with_suffix(\".benchmark.json\").write_text(json.dumps(report, indent=2), encoding=\"utf-8\")\n        print(json.dumps(report, indent=2))\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--video\", required=True)\n    parser.add_argument(\"--repo\", required=True)\n    parser.add_argument(\"--weights\", required=True)\n    parser.add_argument(\"--output\", default=\"housekeeping-map.json\")\n    parser.add_argument(\"--fps\", type=float, default=2, choices=[1, 2, 4])\n    parser.add_argument(\"--profile\", default=\"fast\", choices=[\"fast\", \"detailed\"])\n    parser.add_argument(\"--image-size\", type=int, default=336, choices=[252, 336, 518])\n    parser.add_argument(\"--clip-seconds\", type=float, default=30)\n    options = parser.parse_args()\n    try:\n        run(options)\n    except Exception as error:\n        print(f\"Mapping trial failed: {error}\", file=sys.stderr)\n        raise SystemExit(1)\n"
pathlib.Path('/content/export_map.py').write_text(_exporter_source)
print('Exporter saved for isolated-process inference.')


In [ ]:
import subprocess, sys
PROFILE="fast"
IMAGE_SIZE=518 if PROFILE=="detailed" else 336
FPS=4 if PROFILE=="detailed" else 2
CLIP_SECONDS=30
output='/content/housekeeping-map.json'
result=subprocess.run([sys.executable,'/content/export_map.py','--video',video,'--repo',str(repo),'--weights',weights,'--output',output,'--image-size',str(IMAGE_SIZE),'--fps',str(FPS),'--clip-seconds',str(CLIP_SECONDS),'--profile',PROFILE],capture_output=True,text=True)
print(result.stdout[-4000:])
print(result.stderr[-6000:])
result.check_returncode()
print(pathlib.Path('/content/housekeeping-map.benchmark.json').read_text())


## Download and review

Download the map and benchmark report. Open [Housekeeping setup](https://housekeeping-najib.vercel.app/app?view=setup) as admin → **Experimental 3D home mapping → Import map result**. Rotate the reconstruction and inspect its camera path. Label each area with time range and visible fixtures. Create a draft, review its duties, and apply only when satisfied. Started/submitted work is preserved by the existing setup workflow.

Download any results you need, then use **Runtime → Disconnect and delete runtime** to release the GPU. Colab sessions are temporary; the experiment does not create an always-on service.


In [ ]:
files.download(output)
files.download('/content/housekeeping-map.benchmark.json')
